In [1]:
import pandas as pd

# File path
file_path = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\4.1_Total_Repo_Dataset.csv"

# Read CSV
df = pd.read_csv(file_path)

# CI platform columns in the dataset
ci_platform_cols = [
    'appveyor', 'azure_pipelines', 'bitbucket', 'bitrise', 'circle_ci',
    'cirrus', 'codemagic', 'github_actions', 'gitlab', 'semaphore', 'travis_ci'
]

# Convert wide format to long format
df_long = df.melt(
    id_vars=['full_name'],
    value_vars=ci_platform_cols,
    var_name='ci_platform',
    value_name='yml_count'
)

# Keep only rows with > 0 YAML files for that platform
df_long = df_long[df_long['yml_count'] > 0]

# Group by CI platform and summarize
summary = df_long.groupby('ci_platform').agg(
    num_yml_files=('yml_count', 'sum'),
    num_distinct_repos=('full_name', pd.Series.nunique)
).reset_index()

# Sort by number of repos in descending order
summary_sorted = summary.sort_values(by='num_distinct_repos', ascending=False)

# Display the result
print(summary_sorted)


        ci_platform  num_yml_files  num_distinct_repos
7    github_actions           9724                3258
10        travis_ci           2232                1175
4         circle_ci            280                 273
0          appveyor            208                 107
8            gitlab            119                 103
1   azure_pipelines             37                  35
5            cirrus             30                  23
6         codemagic             21                  21
3           bitrise              9                   9
2         bitbucket              6                   5
9         semaphore              1                   1


In [10]:
import pandas as pd

# --- Config ---
file_path = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\4.1_Total_Repo_Dataset.csv"

# Columns that represent counts of YAML files per CI platform (adjust if your file differs)
CI_PLATFORM_COLS_ALL = [
    'appveyor', 'azure_pipelines', 'bitbucket', 'bitrise', 'circle_ci',
    'cirrus', 'codemagic', 'github_actions', 'gitlab', 'semaphore', 'travis_ci'
]

# Metrics to average
METRICS = [
    'repo_age',          # numeric
    'stargazers_count',  # numeric
    'contributors',      # numeric
    'pull_requests',     # numeric
    'commits_gitapi',    # numeric
    'open_issues_count'  # numeric
]

# --- Load ---
df = pd.read_csv(file_path)
df.columns = [c.strip().lower() for c in df.columns]

# Normalize expected columns
confidence_col = 'instru_t_ci_confidence'
if confidence_col not in df.columns:
    raise KeyError(f"Expected column '{confidence_col}' not found. Found: {list(df.columns)}")

# Use only CI platform columns that exist in the file
ci_platform_cols = [c for c in CI_PLATFORM_COLS_ALL if c in df.columns]

# Compute total YAML files per repo by summing across available CI platform columns
if ci_platform_cols:
    df['total_yml_files_repo'] = df[ci_platform_cols].sum(axis=1, numeric_only=True)
else:
    # Fallback: if your dataset already has a 'total_ymls' column, use it; otherwise default to 0
    df['total_yml_files_repo'] = df.get('total_ymls', 0)

# Ensure metrics are numeric
for m in METRICS:
    if m in df.columns:
        df[m] = pd.to_numeric(df[m], errors='coerce')
    else:
        df[m] = pd.NA

# Standardize blanks in confidence
conf_series = df[confidence_col]
conf_series = conf_series.where(conf_series.notna() & (conf_series.str.strip() != ""), other="")  # "" = blank bucket
df['_confidence_std'] = conf_series

# Group by confidence level INCLUDING blanks
summary = (
    df.groupby('_confidence_std', dropna=False)
      .agg(
          total_repos=('full_name', pd.Series.nunique),        # total repos per confidence (first column)
          total_yml_files=('total_yml_files_repo', 'sum'),     # sum of YAML files per confidence
          avg_repo_age=('repo_age', 'mean'),
          avg_stars=('stargazers_count', 'mean'),
          avg_contributors=('contributors', 'mean'),
          avg_pull_requests=('pull_requests', 'mean'),
          avg_commits_gitapi=('commits_gitapi', 'mean'),
          avg_open_issues=('open_issues_count', 'mean')
      )
      .reset_index()
      .rename(columns={'_confidence_std': 'confidence_level'})
)

# Sort by total_repos descending
summary = summary.sort_values(by='total_repos', ascending=False)

# Round numeric columns for cleaner display
numeric_cols = summary.select_dtypes(include='number').columns
summary[numeric_cols] = summary[numeric_cols].round(2)

print(summary.to_string(index=False))


confidence_level  total_repos  total_yml_files  avg_repo_age  avg_stars  avg_contributors  avg_pull_requests  avg_commits_gitapi  avg_open_issues
                         4044            10772          6.03    1051.67             19.19             237.09             1386.50            50.36
            high          414             1613          6.94    1232.50             30.23             533.09             1828.09            87.60
          medium           48              237          6.48     734.06             23.83             592.85             2348.79            38.96
             low           12               45          6.69     440.58             14.67             257.75              550.92            34.50


In [9]:
import pandas as pd

# File path
file_path = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\4.1_Total_Repo_Dataset.csv"

# Read CSV
df = pd.read_csv(file_path)

# Normalize column names
df.columns = [c.strip().lower() for c in df.columns]

# Columns of interest
confidence_col = 'instru_t_ci_confidence'
language_col = 'language'
repo_col = 'full_name'

# Standardize blanks in confidence
df[confidence_col] = df[confidence_col].fillna("").apply(lambda x: x.strip())

# --- Identify top 5 languages by distinct repo count ---
top5_languages = (
    df.groupby(language_col)[repo_col]
      .nunique()
      .sort_values(ascending=False)
      .head(5)
      .index
      .tolist()
)

# --- Group others into "Other" ---
df[language_col] = df[language_col].apply(lambda x: x if x in top5_languages else "Other")

# --- Build matrix: count distinct repos for each (language, confidence) pair ---
matrix = (
    df.groupby([language_col, confidence_col])[repo_col]
      .nunique()
      .unstack(fill_value=0)
)

# Order rows: top5 + Other
row_order = top5_languages + ["Other"]
matrix = matrix.loc[row_order]

# Optional: add totals row/column
matrix["Total"] = matrix.sum(axis=1)
matrix.loc["Total"] = matrix.sum(axis=0)

print(matrix)


instru_t_ci_confidence        high  low  medium  Total
language                                              
Kotlin                  1318   192    1      12   1523
Java                    1199   141    4      19   1363
Dart                     707    33    2       4    746
C++                      227    11    0       0    238
TypeScript               165    15    3       7    190
Other                    428    22    2       6    458
Total                   4044   414   12      48   4518
